# T4B — Vision domain shift (multilayer FPN / U-Net)

**Lemma D4** · `nuisance="domain_shift"` · [Task doc](../../docs/tasks/t04b-multilayer-vision.md) · [main.pdf](../../main.pdf)

> E1_multiscale rare-5 Cityscapes mIoU **30.75%** (+11.1 pp vs B0 19.68%).

| § | What you do |
|---|-------------|
| 1–4 | Install → load demo → `check_applicability` |
| 5–6 | Estimate $\Sigma_{\text{task}}$ → PMH train → Step 5 on deploy holdout |
| 7–8 | Read paper block → plug in your data |


**Demo note:** Runnable **feature_diff** on a tiny RGB CNN (conv1 + conv2): class-aligned per-layer Gram. Paper GTA5→Cityscapes — §7 scripts.


## 1 — Install


In [ ]:
!pip install -q matching-pmh torch


## 2 — Config & imports


In [ ]:
import os
import torch
from pmh.benchmark.presets import get_preset
from pmh.pytorch_eval import (
    pytorch_demo_loaders,
    pytorch_isotropic_demo_loaders,
    pytorch_multilayer_vision_demo_loaders,
    pytorch_sequence_demo_loaders,
)
from pmh import PMHConfig, PMHTrainer, evaluate_robust_fit, check_applicability, suggest_nuisance
from pmh.adoption import RECIPE_ONE_LINER, format_recipe_banner

QUICK = os.environ.get("PMH_QUICK", "").lower() in ("1", "true", "yes")
EPOCHS = 2 if QUICK else 6
SEED = 0
print(RECIPE_ONE_LINER)


## 3 — Load demo data


In [ ]:
preset = get_preset("t4_domain_d4")
N = 200 if QUICK else 500
bundle = pytorch_multilayer_vision_demo_loaders(n=N, batch_size=32, seed=SEED)
model = bundle.model
hook, head = bundle.encoder, bundle.head
train_loader, src_loader, tgt_loader, val_loader = (
    bundle.train_loader, bundle.source_batches, bundle.target_batches, bundle.val_loader,
)
print("RGB multilayer demo", bundle.n_classes, "classes")


## 4 — Scope (applicability)


In [ ]:
from pmh import check_applicability, suggest_nuisance

print(suggest_nuisance(has_source_labels=True, has_target_domain=True))
app = check_applicability(stack="pytorch", has_target_domain=True)
print(app.summary())
print("suggested nuisance:", app.suggested_nuisance, "(expect 'domain_shift')")


## 5 — Estimate $\Sigma_{\text{task}}$ + PMH train


In [ ]:
import copy
from pmh import PMHTrainer, PMHConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m = copy.deepcopy(model).to(device)
layer_names = ("conv1", "conv2")
forward_features = m.forward_features
trainer = PMHTrainer(
    m, hook=bundle.encoder, head=m.head, nuisance="domain_shift", rank=preset.default_rank, pmh_config=preset.pmh_config,
    train_mode="feature_diff", forward_features=forward_features, layer_names=layer_names,
    head_layer=layer_names[-1], device=device,
)
sigmas = trainer.estimate_multilayer(src_loader, tgt_loader, max_batches=10 if QUICK else 30)
print("per-layer sigma shapes:", {k: tuple(v.shape) for k, v in sigmas.items()})
trainer.fit(
    train_loader, source_batches=src_loader, target_batches=tgt_loader,
    epochs=EPOCHS, max_steps_per_epoch=8 if QUICK else None,
)
print("feature_diff train done; preflight", trainer.artifact_.preflight)


## 6 — Step 5 (deploy holdout)


In [ ]:
from pmh import evaluate_robust_fit

report = evaluate_robust_fit(
    m, train_loader, val_loader,
    source_batches=src_loader, target_batches=tgt_loader,
    hook=bundle.encoder, head=m.head, nuisance="domain_shift", rank=preset.default_rank, 
    pmh_config=preset.pmh_config, epochs=max(2, EPOCHS - 2), include_falsification=True, seed=SEED,
)
print(report.summary())
if hasattr(report, "baseline_metric"):
    print("deploy holdout — baseline:", report.baseline_metric, "pmh:", report.pmh_metric)


### Golden path (deploy QA)


In [ ]:
from pmh import try_pmh

report = try_pmh(
    m, train_loader, val_loader,
    source_batches=src_loader, target_batches=tgt_loader,
    hook=bundle.encoder, head=m.head, epochs=2 if QUICK else 4,
)
print(report.deploy_summary())


## 7 — Paper results


Paper headlines: [main.pdf](../../main.pdf) · [findings.html](../../docs/findings.html)

- **Cityscapes rare-5 mIoU:** E1_multiscale +11.1 pp mIoU.
- **Build rare-5 training subset:** 
- **Pixel-aligned per-layer TDI:** 


## 8 — Your pipeline


Swap demo loaders for your `train_loader`, `source_batches`, `target_batches`, and deploy holdout. Hook the backbone before your task head.
